In [53]:
import requests
import pandas as pd
import io
import time
import matplotlib.pyplot as plt
from geopy.distance import geodesic # Import geodesic

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
#Get API from USGS
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

start_year = 2020
end_year = 2025  # Adjust to the latest date
min_magnitude = 4.0  # Adjust as needed

# Define batch sizes based on data volume
batch_sizes = {
    (2024, 2025): 1,
}

#store dataframes
all_data = []

# Loop through different batch sizes and store in all_data
for (start, end), batch_size in batch_sizes.items():
    for year in range(start, end + 1, batch_size):
        start_time = f"{year}-01-01"
        end_time = f"{min(year + batch_size - 1, end_year)}-12-31"

        params = {
            "format": "csv",
            "starttime": start_time,
            "endtime": end_time,
            "minmagnitude": min_magnitude,
            "maxmagnitude": 10.0,
            "orderby": "time"
        }

        print(f"Fetching data from {start_time} to {end_time}...")

        response = requests.get(url, params=params)

        if response.status_code == 200:
            # Convert response content to dataframe
            df = pd.read_csv(io.StringIO(response.text))
            all_data.append(df)

            print(f"✅ Data fetched for {start_time} to {end_time} ({len(df)} records)")
        else:
            print(f"❌ Failed for {start_time} to {end_time}: {response.status_code}")

        # Sleep to avoid hitting API rate limits
        time.sleep(2)

# Merge all data
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv("earthquake_data_global.csv", index=False)
    print("✅ All data saved: earthquake_data_global.csv")
else:
    print("❌ No data collected.")




Fetching data from 2024-01-01 to 2024-12-31...
✅ Data fetched for 2024-01-01 to 2024-12-31 (14132 records)
Fetching data from 2025-01-01 to 2025-12-31...
✅ Data fetched for 2025-01-01 to 2025-12-31 (3675 records)
✅ All data saved: earthquake_data_global.csv


In [56]:
# Load the full earthquake dataset
df_eq = pd.read_csv("earthquake_data_global.csv")

# Define Myanmar's geographic boundaries (approximate lat/lon range)
myanmar_bounds = (16.0, 28.0, 92.0, 101.0)

# Filter for earthquakes within Myanmar
df_myanmar = df_eq[
    (df_eq["latitude"] >= myanmar_bounds[0]) & (df_eq["latitude"] <= myanmar_bounds[1]) &
    (df_eq["longitude"] >= myanmar_bounds[2]) & (df_eq["longitude"] <= myanmar_bounds[3])
]

# Save the Myanmar-specific earthquake dataset
df_myanmar.to_csv("earthquake_data_myanmar.csv", index=False)

print(f"✅ Filtered dataset saved: earthquake_data_myanmar.csv ({len(df_myanmar)} records)")
df_myanmar.head()


✅ Filtered dataset saved: earthquake_data_myanmar.csv (125 records)


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
306,2024-12-23T17:57:25.649Z,19.5557,95.0613,85.383,4.9,mb,61.0,83.0,1.647,0.52,...,2025-03-08T22:39:09.040Z,"26 km NW of Myaydo, Myanmar",earthquake,7.60,7.186,0.055,103.0,reviewed,us,us
429,2024-12-20T13:13:08.914Z,23.6675,94.0488,69.573,4.4,mb,58.0,136.0,2.578,0.63,...,2025-02-27T17:30:53.040Z,"36 km W of Mawlaik, Myanmar",earthquake,7.72,7.246,0.081,44.0,reviewed,us,us
503,2024-12-18T16:46:43.648Z,26.1308,100.7419,10.000,4.3,mb,27.0,81.0,4.937,1.00,...,2025-02-27T17:30:48.040Z,"20 km NE of Pianjiao, China",earthquake,6.91,1.851,0.129,17.0,reviewed,us,us
717,2024-12-14T01:05:29.137Z,25.9364,96.5936,10.000,4.4,mb,40.0,111.0,4.954,0.46,...,2025-02-18T20:02:58.040Z,"36 km SW of Sarupathar, India",earthquake,7.37,1.906,0.101,28.0,reviewed,us,us
810,2024-12-11T11:53:29.138Z,22.5954,93.8612,55.118,4.4,mb,41.0,160.0,1.695,0.65,...,2025-02-18T20:02:55.040Z,"26 km ESE of Hakha, Myanmar",earthquake,11.35,9.176,0.099,29.0,reviewed,us,us


Load population data and calculate affected population

In [67]:


# Load the dataframe from your Google Drive
df_population_mm = pd.read_csv('/content/MyanmarPop_2014_Township.csv')

df_pop = df_population_mm
df_pop.head()


,name_tsp,hh_total,hh_male,hh_female,lat,lon
0,Myitkyina,52591,36101,16490,25.38749,97.39036
1,Waingmaw,22860,16034,6826,25.40319,98.11739
2,Injangyang,285,232,53,25.82991,97.72671
3,Tanai,8528,7050,1478,26.69478,96.20940
4,Chipwi,3306,2672,634,25.88635,98.12976


In [ ]:
# Function to estimate affected population for multiple earthquakes at once
def estimate_affected_population_vectorized(df_eq, df_pop):
    # Convert lat/lon to NumPy arrays for fast operations
    eq_lats = df_eq["latitude"].to_numpy()
    eq_lons = df_eq["longitude"].to_numpy()
    eq_mags = df_eq["mag"].to_numpy()

    pop_lats = df_pop["lat"].to_numpy()
    pop_lons = df_pop["lon"].to_numpy()
    pop_totals = df_pop["hh_total"].to_numpy()

    affected_pops = []

    for eq_lat, eq_lon, eq_mag in zip(eq_lats, eq_lons, eq_mags):
        # Compute distances for all townships at once
        distances_km = np.array([geodesic((eq_lat, eq_lon), (pl, plon)).km for pl, plon in zip(pop_lats, pop_lons)])

        # Define weights based on distances
        weights = np.where(distances_km <= 50, 1,
                   np.where(distances_km <= 100, 0.5,
                   np.where(distances_km <= 200, 0.2, 0)))

        # Compute affected population
        affected_pop = np.sum(pop_totals * weights * (eq_mag / 7.0))
        affected_pops.append(int(affected_pop))

    df_eq["Estimated_Affected_Pop"] = affected_pops
    return df_eq

# Apply function
df_eq = estimate_affected_population_vectorized(df_eq, df_pop)

# Save results
df_eq.to_csv("earthquake_affected_population_optimized.csv", index=False)
print("✅ Optimized estimated affected population saved in 'earthquake_affected_population_optimized.csv'")

# Display results
df_eq.tail()

In [63]:
# Function to estimate affected population based on distance
def estimate_affected_population(eq_lat, eq_lon, eq_mag):
    affected_pop = 0

    for _, row in df_pop_filtered.iterrows():
        region_lat = row["lat"]
        region_lon = row["lon"]
        total_pop = row["hh_total"]

        # Calculate distance (km) from earthquake to the region center
        distance_km = geodesic((eq_lat, eq_lon), (region_lat, region_lon)).km

        # Define an impact weight based on distance
        if distance_km <= 50:
            weight = 1  # High impact (full population affected)
        elif distance_km <= 100:
            weight = 0.5  # Medium impact
        elif distance_km <= 200:
            weight = 0.2  # Low impact
        else:
            weight = 0  # No significant impact

        # Estimate affected population
        affected_pop += total_pop * weight * (eq_mag / 7.0)  # Scale by magnitude

    return int(affected_pop)

# Apply function to all earthquakes
df_eq["Estimated_Affected_Pop"] = df_eq.apply(
    lambda row: estimate_affected_population(row["latitude"], row["longitude"], row["mag"]), axis=1
)

# Save results
df_eq.to_csv("earthquake_affected_population_filtered.csv", index=False)
print("✅ Estimated affected population saved in 'earthquake_affected_population_filtered.csv'")

# Display results
df_eq.tail()


KeyboardInterrupt: 

Weighted Affected People 2024-2025


In [ ]:
import folium
import pandas as pd
import numpy as np
import ipywidgets as widgets
from folium.plugins import HeatMap
from geopy.distance import geodesic
from IPython.display import display
from datetime import datetime

# Load earthquake data
df_eq = pd.read_csv("earthquake_data_myanmar.csv.csv")

# Convert 'time' column to datetime and remove timezone
df_eq["time"] = pd.to_datetime(df_eq["time"], errors="coerce")
df_eq["time"] = df_eq["time"].dt.tz_localize(None)

# Filter for 2024 and 2025 earthquakes
df_eq_filtered = df_eq[df_eq["time"].dt.year.isin([2024, 2025])].copy()

# Load population dataset
df_pop_mm = pd.read_csv('/content/drive/MyDrive/GoogleColab/MyanmarPop_2014_Township.csv')

# 🔹 **Population Estimation Function (No Region Filter)**
def estimate_affected_population(eq_lat, eq_lon, eq_mag):
    affected_pop = 0
    selected_pop_data = df_pop_mm.copy()  # Use entire dataset

    # Ensure numeric types
    selected_pop_data["Latitude"] = pd.to_numeric(selected_pop_data["Latitude"], errors="coerce")
    selected_pop_data["Longitude"] = pd.to_numeric(selected_pop_data["Longitude"], errors="coerce")
    selected_pop_data["Both sexes"] = pd.to_numeric(selected_pop_data["Both sexes"], errors="coerce")

    print(f"\n🔍 Earthquake at ({eq_lat}, {eq_lon}) M{eq_mag}")
    print(f"📌 Analyzing {len(selected_pop_data)} population points...")

    for _, row in selected_pop_data.iterrows():
        region_lat = row["Latitude"]
        region_lon = row["Longitude"]
        total_pop = row["Both sexes"]

        # Skip invalid data
        if pd.isna(region_lat) or pd.isna(region_lon) or pd.isna(total_pop):
            continue

        # Calculate distance from earthquake to region center
        distance_km = geodesic((eq_lat, eq_lon), (region_lat, region_lon)).km

        # Define dynamic buffer based on earthquake magnitude
        max_radius = eq_mag * 15  # Example: M6.5 -> 65 km radius

        if distance_km <= max_radius:
            # Use an exponential decay function to reduce impact with distance
            weight = np.exp(-distance_km / (max_radius / 2))
            magnitude_factor = (eq_mag / 7.0) ** 2  # Scale by magnitude

            affected_pop += total_pop * weight * magnitude_factor

            print(f"🌍 Region at ({region_lat}, {region_lon}) | Pop: {total_pop}")
            print(f"   ➡️ Distance: {distance_km:.2f} km, Weight: {weight:.4f}, Mag Factor: {magnitude_factor:.4f}")

    print(f"🎯 Estimated Affected Population: {int(affected_pop)}\n")
    return int(affected_pop)


# 🔹 **Map Visualization Function**
def plot_earthquake_map(start_date=None, end_date=None):
    # Default dates: 2025-01-01 to today
    if start_date is None:
        start_date = datetime(2025, 1, 1)
    if end_date is None:
        end_date = datetime.today()

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # Filter earthquakes by selected date range
    filtered_eq = df_eq_filtered[(df_eq_filtered["time"] >= start_date) & (df_eq_filtered["time"] <= end_date)].copy()

    if filtered_eq.empty:
        print("⚠️ No earthquakes found in the selected time range.")
        return None

    # Compute affected population for each earthquake (without region filtering)
    filtered_eq["Weighted_Affected_Pop"] = filtered_eq.apply(
        lambda row: estimate_affected_population(row["latitude"], row["longitude"], row["mag"]),
        axis=1
    )

    # Ensure affected population is not all zeros
    print("📊 Min Affected Pop:", filtered_eq["Weighted_Affected_Pop"].min())
    print("📊 Max Affected Pop:", filtered_eq["Weighted_Affected_Pop"].max())

    # Create Folium Map
    myanmar_map = folium.Map(location=[21.0, 96.0], zoom_start=6)

    # 🔹 **Add Population Heatmap**
    heat_data = [
        [row["Latitude"], row["Longitude"], row["Both sexes"]]
        for _, row in df_pop_mm.iterrows()
    ]

    if heat_data:
        HeatMap(heat_data, min_opacity=0.4, radius=20, blur=10, max_zoom=1).add_to(myanmar_map)

    # 🔹 **Add Earthquake Markers**
    for _, row in filtered_eq.iterrows():
        weighted_population = row["Weighted_Affected_Pop"]

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=max(5, np.log1p(weighted_population) / 2),
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=min(0.9, weighted_population / 1e6) if weighted_population > 0 else 0.3,
            popup=(
                f"<b>Date:</b> {row['time'].date()}<br>"
                f"<b>Magnitude:</b> {row['mag']}<br>"
                f"<b>Estimated Affected Population:</b> {int(weighted_population):,}"
            ),
        ).add_to(myanmar_map)

    display(myanmar_map)


# 🔹 **Interactive Widget for Filtering (Only Date Selection)**
default_start_date = datetime.today()
default_end_date = datetime.today()

date_picker = widgets.DatePicker(
    description="Start Date",
    value=default_start_date,  # Default to 2025-01-01
    layout={"width": "50%"}
)

end_date_picker = widgets.DatePicker(
    description="End Date",
    value=default_end_date,  # Default to today's date
    layout={"width": "50%"}
)

widgets.interactive(
    plot_earthquake_map,
    start_date=date_picker,
    end_date=end_date_picker
)


interactive(children=(DatePicker(value=datetime.datetime(2025, 4, 1, 11, 3, 29, 194535), description='Start Da…